# 4. Aquitectura de la red 

- Se realiza la construcción de las **estructura de la red** siguiendo el enfoque de Yildiz et al. (2025)

:::{figure} ./images/EsquemaCapa.png
:alt: Estructura del modelo
:width: 90%
:align: center

Figura 2. Estructura del modelo ConvLSTM.
:::

## 4.1. Pasos

1. Se cargan los tensores de entrada y salida ya construidos
2. Se define la arquitectura de la red ConvLSTM
3. Se separan los datos en entrenamiento, validación y prueba con un criterio temporal
4. Entrenamiento del modelo, evaluación del desempeño y visualización de predicciones

El objetivo es aprender la relación:

**(t-1, t) → t+1**



## 4.2. Definición de la arquitectura

La red propuesta combina capas convolucionales espaciales con una capa recurrente espacial-temporal ConvLSTM. De este modo, el modelo busca extraer primero rasgos espaciales por cada instante de entrada y, luego, aprender la dependencia temporal entre esos estados.

In [ ]:
from tensorflow.keras import layers, models

# Capa Input
inp = layers.Input(shape=input_shape)

# Capa Conv2D
x = layers.TimeDistributed(
    layers.Conv2D(96, (3, 3), activation="relu", padding="same")
)(inp)

# Capa MaxPooling
#x = layers.TimeDistributed(
    #layers.MaxPooling2D(pool_size=(2, 2))
#)(x)

# Capa ConvLSTM2D
x = layers.ConvLSTM2D(
    filters=48,
    kernel_size=(3, 3),
    activation="relu",
    recurrent_activation="sigmoid",
    padding="same",
    return_sequences=False
)(x)

# Capa MaxPooling
#x = layers.MaxPooling2D(pool_size=(2, 2))(x)

# Capa Conv2D
x = layers.Conv2D(48, (3, 3), activation="relu", padding="same")(x)

# Capa Upsampling
#x = layers.UpSampling2D(size=(2, 2))(x)

# Capa Conv2D
x = layers.Conv2D(96, (3, 3), activation="relu", padding="same")(x)

# Capa Upsampling
#x = layers.UpSampling2D(size=(2, 2))(x)

# Capa de salida
out = layers.Conv2D(1, (3, 3), activation="linear", padding="same")(x)

model = models.Model(inputs=inp, outputs=out)
model.summary()


## 3.3. Construcción de muestras supervisadas

In [ ]:
X_list = []
Y_list = []
meta_list = []

for tile_id, g in df_tiles.groupby("tile_id"):
    idxs = g.index.to_numpy()

    for k in range(len(idxs) - TIME_STEP):
        idx_input = idxs[k : k + TIME_STEP]
        idx_target = idxs[k + TIME_STEP]

        x_sample = X_tiles[idx_input]   # (TIME_STEP, 68, 68)
        y_sample = X_tiles[idx_target]  # (68, 68)

        X_list.append(x_sample)
        Y_list.append(y_sample)

## 3.4. Forma final de los datos

Al finalizar, los tensores quedan con la siguiente estructura conceptual:

- **`X`**: `(n_samples, TIME_STEP, TILE_SIZE, TILE_SIZE, 1)`
- **`Y`**: `(n_samples, TILE_SIZE, TILE_SIZE, 1)`

Esto significa que cada muestra de entrada contiene una pequeña secuencia temporal de mapas espaciales, y cada salida corresponde al mapa espacial del tiempo siguiente.